# 08 - Ablations, probes and measured compute
1. Ablations on addition (`configs/ablations.yaml`, seed 0): no recall, r = 2, width 64.
2. Linear probes: is "position i is still pending" (t < rho_i) linearly decodable from h_i(t), and
   does a probe fitted on ID transfer to OOD-depth and OOD-both?
3. Measured FLOPs (`torch.utils.flop_counter.FlopCounterMode`) and CUDA-event latency for every
   stopping rule, using online early exit (halted instances leave the batch).

In [1]:
# parameters
ABL_CONFIG = "configs/ablations.yaml"
MAIN_CONFIG = "configs/main_addition.yaml"
GATES_CONFIG = "configs/main_gates.yaml"
PROBE_ROWS = 150_000
LATENCY_N = 512

In [2]:
%matplotlib inline
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from glwm2 import PROJECT_ROOT, RESULTS_DIR, PAPER_DIR
from glwm2.eval.plotting import set_style
set_style()
WRITTEN = []  # files this notebook writes

In [3]:
from glwm2.eval.results import configs_from_yaml, finished, metric_rows
from glwm2.eval.analysis import ENTRIES, entry_rows
from glwm2.eval.plotting import savefig
from glwm2.utils.io import write_json
abl = configs_from_yaml(PROJECT_ROOT / ABL_CONFIG)
main = [c for c in configs_from_yaml(PROJECT_ROOT / MAIN_CONFIG)]
print(f"ablations: {len(finished(abl))}/{len(abl)} finished")
ref = [c for c in finished(main) if c.method in ("base", "done_joint") and c.seed in (0, 1)]
for c in ref:
    c.tags = {"ablation": "main"}
A = metric_rows(finished(abl) + ref)
if len(A):
    tag = {c.run_id(): c.tags.get("ablation") for c in finished(abl) + ref}
    A["ablation"] = A.run_id.map(tag)
    rule_of = {"base": ["oracle_stable", "max_softmax", "done_prop_post"], "done_joint": ["done_joint"]}
    A = A[[r in rule_of.get(m, []) for m, r in zip(A.method, A.rule)]]
    abl_tab = A.pivot_table(index=["ablation", "method", "rule"], columns="split", values="acc", aggfunc="mean")[["id_test", "ood_depth", "ood_length", "ood_both"]]
    display(abl_tab.round(3))

ablations: 3/3 finished


split                                id_test  ood_depth  ood_length  ood_both
ablation  method     rule                                                    
main      base       done_prop_post    1.000      0.998       1.000     0.996
                     max_softmax       1.000      1.000       1.000     0.998
                     oracle_stable     1.000      1.000       1.000     0.996
          done_joint done_joint        1.000      1.000       1.000     0.990
no_recall base       done_prop_post    1.000      1.000       1.000     0.072
                     max_softmax       1.000      1.000       1.000     0.075
                     oracle_stable     1.000      1.000       1.000     0.006
r_2       base       done_prop_post    0.999      1.000       1.000     0.997
                     max_softmax       1.000      1.000       1.000     1.000
                     oracle_stable     1.000      1.000       1.000     1.000
width_64  base       done_prop_post    0.996      0.996       0.999     0.562
                     max_softmax       1.000      1.000       1.000     0.480
                     oracle_stable     1.000      1.000       1.000     0.952

In [4]:
# linear probes for the pending state (seed-0 base model)
from glwm2.train.trainer import load_model
from glwm2.data.splits import load_split
from glwm2.eval.probes import collect_probe_data, fit_linear_probe, probe_scores
c0 = next(c for c in finished(main) if c.method == "base" and c.seed == 0)
model = load_model(c0)
probe_res = {}
tr = collect_probe_data(model, "addition", load_split("addition", "id_val"), t_max=26, max_rows=PROBE_ROWS)
probe = fit_linear_probe(tr["h"], tr["pending"])
for split, T in [("id_test", 26), ("ood_depth", 36), ("ood_both", 144)]:
    te = collect_probe_data(model, "addition", load_split("addition", split), t_max=T, max_rows=PROBE_ROWS, seed=1)
    probe_res[split] = probe_scores(probe, te["h"], te["pending"])
pd.DataFrame(probe_res).T.round(3)

,acc,auc,base_rate
id_test,1.0,1.0,0.047
ood_depth,1.0,1.0,0.152
ood_both,1.0,1.0,0.139


In [5]:
# measured FLOPs per instance for every rule: encode + steps x (step + decode + head) at the mean length of each split
from glwm2.eval.flops import backbone_flops, module_flops
from glwm2.train.pipeline import get_posthoc_heads
heads = get_posthoc_heads(model, c0, "cuda")
rows_main = entry_rows(finished(main) + [c for c in finished(configs_from_yaml(PROJECT_ROOT / GATES_CONFIG)) if c.task == "addition"])
flop_rows = []
cpu_model = load_model(c0, "cpu")
for split in ["id_test", "ood_depth", "ood_length", "ood_both"]:
    d = load_split("addition", split)
    P = int(d["length"].float().mean().round()) + 1
    x = torch.randint(0, 100, (1, P))
    f = backbone_flops(cpu_model.backbone, x)
    h = cpu_model.backbone.init_state(cpu_model.backbone.encode(x))
    head_f = {"done_post": module_flops(heads["done_post"].cpu(), h), "done_prop_post": module_flops(heads["done_prop_post"].cpu(), h),
              "trm_post": module_flops(heads["trm_post"].cpu(), h, None)}
    for e, (method, rule, fam, label) in ENTRIES.items():
        sub = rows_main[(rows_main.entry == e) & (rows_main.split == split)]
        if sub.empty: continue
        steps = sub.mean_steps.mean()
        per_step = f["step"] + f["decode"] + head_f.get(e, 0)
        flop_rows.append(dict(split=split, entry=e, family=fam, mean_steps=steps, kflops=(f["encode"] + steps * per_step) / 1e3,
                              acc=sub.acc.mean()))
for hh in heads.values():
    hh.cuda()
flops_df = pd.DataFrame(flop_rows)
display(flops_df.pivot_table(index="entry", columns="split", values="kflops").reindex([e for e in ENTRIES if e in set(flops_df.entry)]).round(1))

W1013 14:44:30.055000 26136 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


split,id_test,ood_both,ood_depth,ood_length
entry,,,,
fixed_T,34411.5,423526.4,42352.6,428820.5
oracle_lb,17525.8,3191525.5,57159.1,234650.6
oracle_stable,9888.2,1708187.9,20672.3,188595.2
oracle_iter,9823.6,1673352.8,17899.3,188595.2
max_softmax,17525.8,3200207.8,57159.1,234693.4
entropy,17525.8,3191906.7,57159.1,234650.6
stability,13245.0,1714201.9,22136.7,231477.3
kl,13870.4,1720321.9,22531.6,240804.1
hidden_change,21324.8,177267.0,59637.8,182398.8


In [6]:
# CUDA-event latency with online early exit on OOD-depth (batch of LATENCY_N instances)
from glwm2.eval.latency import cuda_time_ms
from glwm2.readouts.online import run_online
from glwm2.data.splits import load_split
d = load_split("addition", "ood_depth", "cuda")
x = d["x"][:LATENCY_N]
thr = {r.rule: r.threshold for r in metric_rows([c0]).itertuples() if r.split == "id_val"}
T_MAX = int(d["depth"].max()) + 16
def conf_rule(tau):
    def f(o):
        p = torch.softmax(o["logits"].float(), -1).max(-1).values
        return torch.where(o["mask"], p, torch.ones_like(p)).min(1).values > tau
    return f
lat = {
    "fixed_T (val-selected)": cuda_time_ms(lambda: run_online(model, x, int(thr["fixed_T"]), None)),
    "fixed_T = T_max": cuda_time_ms(lambda: run_online(model, x, T_MAX, None)),
    "max_softmax": cuda_time_ms(lambda: run_online(model, x, T_MAX, conf_rule(float(thr["max_softmax"])))),
    "done_post": cuda_time_ms(lambda: run_online(model, x, T_MAX, None, head=heads["done_post"], head_thr=float(thr["done_post"]))),
    "done_prop_post": cuda_time_ms(lambda: run_online(model, x, T_MAX, None, head=heads["done_prop_post"], head_thr=float(thr["done_prop_post"]))),
}
lat_df = pd.DataFrame(lat).T; lat_df.round(2)

,median_ms,q25_ms,q75_ms
fixed_T (val-selected),7.38,6.39,8.37
fixed_T = T_max,23.84,20.40,28.51
max_softmax,19.75,18.91,21.14
done_post,23.77,21.71,26.24
done_prop_post,28.34,25.54,34.11


In [7]:
fig, ax = plt.subplots(figsize=(3.4, 2.4))
sub = flops_df[flops_df.split == "ood_depth"]
from glwm2.eval.plotting import FAMILY_COLORS
for r_ in sub.itertuples():
    ax.scatter(r_.kflops, r_.acc, color=FAMILY_COLORS[r_.family], s=14)
    ax.annotate(r_.entry, (r_.kflops, r_.acc), fontsize=4.5)
ax.set_xscale("log"); ax.set_xlabel("measured kFLOPs per instance"); ax.set_ylabel("accuracy (OOD-depth)")
WRITTEN.append(savefig(fig, "fig8_compute"))

# LaTeX: probes, ablations (OOD accuracy of the stable oracle and max-softmax) and latency
lines = [r"\begin{table}[h]", r"\centering",
         r"\caption{Left: linear probe for the pending state (fitted on ID-val, seed-0 base model on addition). "
         r"Middle: ablations on addition (seed 0; accuracy in \%). Right: CUDA-event latency for 512 OOD-depth "
         r"instances with online early exit (median ms).}",
         r"\label{tab:probes}", r"\small",
         r"\begin{tabular}{lrr}", r"\toprule", r"Split & Acc. & AUC \\", r"\midrule"]
for s, v in probe_res.items():
    lines.append(f"{s.replace('_', '-')} & {100 * v['acc']:.1f} & {v['auc']:.3f} \\\\")
lines += [r"\bottomrule", r"\end{tabular}\quad"]
if len(A):
    t = abl_tab.reset_index()
    t = t[t.rule.isin(["oracle_stable", "max_softmax"])]
    lines += [r"\begin{tabular}{llrr}", r"\toprule", r"Ablation & Rule & OOD-depth & OOD-both \\", r"\midrule"]
    for r_ in t.itertuples():
        lines.append(f"{str(r_.ablation).replace('_', ' ')} & {r_.rule.replace('_', ' ')} & {100 * r_.ood_depth:.1f} & {100 * r_.ood_both:.1f} \\\\")
    lines += [r"\bottomrule", r"\end{tabular}\quad"]
lines += [r"\begin{tabular}{lr}", r"\toprule", r"Rule & ms \\", r"\midrule"]
for k_, v in lat.items():
    lines.append(f"{k_.replace('_', ' ')} & {v['median_ms']:.1f} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
p = PROJECT_ROOT / "paper" / "tables" / "probes_ablations.tex"
p.write_text("\n".join(lines) + "\n", encoding="utf-8"); WRITTEN.append(p)

WRITTEN.append(write_json(RESULTS_DIR / "compute_probes_ablations.json", {
    "probes": probe_res, "flops": flops_df.to_dict(orient="records"), "latency_ms": lat_df.reset_index().to_dict(orient="records"),
    "ablations": (abl_tab.reset_index().to_dict(orient="records") if len(A) else [])}))

In [8]:
print("Files written by this notebook:")
for p in WRITTEN:
    print(" ", Path(p).relative_to(PROJECT_ROOT))

Files written by this notebook:
  paper\figures\fig8_compute.pdf
  paper\tables\probes_ablations.tex
  results\compute_probes_ablations.json
